In [6]:
import re
import csv
from time import time
import numpy as np
import pandas as pd
import warnings
from scipy.sparse import hstack

# ML Imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

warnings.filterwarnings("ignore")

In [7]:
# Data loading and preprocessing functions
COLS = ["id","label","statement","subjects","speaker","job_title",
        "state_info","party_affiliation","barely_true_cnt","false_cnt",
        "half_true_cnt","mostly_true_cnt","pants_on_fire_cnt","context","justification"]

data_path = '../data/'

def read_tsv(path):
    """Load TSV data with proper handling of quotes and escape characters"""
    return pd.read_csv(path, sep="\t", header=None, names=COLS,
                       engine="python", quoting=csv.QUOTE_NONE, escapechar="\\",
                       on_bad_lines="skip")

def text_of(r):
    """Combine statement, context, and justification into single text"""
    return " ".join([str(r.get("statement","")), str(r.get("context","")), str(r.get("justification",""))]).strip()

def first_subject(s):
    """Extract first subject from subjects field"""
    parts = re.split(r"[;,]", s) if isinstance(s,str) else []
    return parts[0].strip().lower() if parts and parts[0].strip() else "unknown"

df_tr = read_tsv(data_path + 'train2.tsv')
df_te = read_tsv(data_path + 'test2.tsv')
df_va = read_tsv(data_path + 'val2.tsv')

print("Data loading functions defined!")

Data loading functions defined!


In [8]:
TOKEN_RE = re.compile(r"[A-Za-z]+")
EVIDENCE_PATTERNS = [
    r"\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b", r"\b(19|20)\d{2}\b", r"%", r"https?://",
    r"\"[^\"]+\"", r"\baccording to\b", r"\breport(ed|s)? by\b|\bstudy\b|\bsurvey\b"
]
EVIDENCE_RE = [re.compile(pat, flags=re.I) for pat in EVIDENCE_PATTERNS]

def evidence_anchors(text):
    s = str(text)
    hits = sum(len(r.findall(s)) for r in EVIDENCE_RE)
    toks = max(1, len(TOKEN_RE.findall(s)))
    dens = np.log1p(hits) / max(10, toks)
    return float(np.clip(dens, 0.0, 0.2))

tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1,2))

# Combine Statement + Context
train_text = df_tr['statement'].astype(str) + " " + df_tr['context'].astype(str)
test_text = df_te['statement'].astype(str) + " " + df_te['context'].astype(str)

# Create TF-IDF Matrices
X_train_tfidf = tfidf.fit_transform(train_text)
X_test_tfidf = tfidf.transform(test_text)

# Create Evidence Features
X_train_ev = df_tr['statement'].apply(evidence_anchors).values.reshape(-1, 1)
X_test_ev = df_te['statement'].apply(evidence_anchors).values.reshape(-1, 1)

# Stack Features
X_train_final = hstack([X_train_tfidf, X_train_ev])
X_test_final = hstack([X_test_tfidf, X_test_ev])

# --- Target Setup ---
SCORE_MAP = {
    "pants-fire": 5, 
    "false": 4, 
    "barely-true": 3, 
    "half-true": 2, 
    "mostly-true": 1, 
    "true": 0
}

# Map labels to scores
train_scores = df_tr['label'].map(SCORE_MAP).fillna(0)
test_scores = df_te['label'].map(SCORE_MAP).fillna(0)

# THRESHOLD = 2.5 means:
# Scores 0, 1, 2 (True, Mostly-True, Half-True) -> 0 (Neutral)
# Scores 3, 4, 5 (Barely-True, False, Pants-Fire) -> 1 (Sensational)
THRESHOLD = 2.5 

y_train_binary = df_tr['label'].map(SCORE_MAP).fillna(0).apply(lambda x: 1 if x >= THRESHOLD else 0)
y_test_binary = df_te['label'].map(SCORE_MAP).fillna(0).apply(lambda x: 1 if x >= THRESHOLD else 0)

In [9]:
names = [
    "Nearest Neighbors", "Linear SVM", "RBF SVM", 
    "Decision Tree", "Random Forest", "AdaBoost", "Multinomial NB"
]

classifiers = [
    KNeighborsClassifier(3),
    SVC(kernel="linear", C=0.025, class_weight="balanced"),
    SVC(gamma=2, C=1, class_weight="balanced"),
    DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    RandomForestClassifier(
        max_depth=5, n_estimators=100, max_features='sqrt',
        class_weight='balanced', random_state=42
    ),
    AdaBoostClassifier(algorithm='SAMME', random_state=42),
    MultinomialNB()
]

print(f"{'Classifier':<20} {'CV Acc':<10} {'CV F1':<10} {'Time (s)':<10}")
print("=" * 60)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'f1_macro']

results = []
best_name = None
best_cv_f1 = -1.0
best_pipe = None

for name, clf in zip(names, classifiers):
    start_time = time()
    
    pipe = make_pipeline(MaxAbsScaler(), clf)
    
    cv_results = cross_validate(
        pipe, X_train_final, y_train_binary,
        cv=cv, scoring=scoring_metrics, n_jobs=-1
    )
    
    cv_acc = 100.0 * cv_results['test_accuracy'].mean()
    cv_f1 = cv_results['test_f1_macro'].mean()
    training_time = time() - start_time
    
    print(f"{name:<20} {cv_acc:>8.2f}% {cv_f1:>9.4f} {training_time:>8.2f}")
    
    results.append({
        "name": name,
        "cv_acc": cv_acc,
        "cv_f1": cv_f1,
        "time": training_time,
    })
    
    # choose best by CV F1
    if cv_f1 > best_cv_f1:
        best_cv_f1 = cv_f1
        best_name = name
        best_pipe = pipe

print("=" * 60)
print(f"Best classifier by CV F1: {best_name} (CV F1 = {best_cv_f1:.4f})")

# Final evaluation on test set
best_pipe.fit(X_train_final, y_train_binary)
y_pred_test = best_pipe.predict(X_test_final)

test_acc = 100.0 * accuracy_score(y_test_binary, y_pred_test)
test_f1  = f1_score(y_test_binary, y_pred_test, average='macro')

print(f"Test Accuracy of best model: {test_acc:.2f}%")
print(f"Test Macro-F1 of best model: {test_f1:.4f}")
print("=" * 60)

Classifier           CV Acc     CV F1      Time (s)  
Nearest Neighbors       56.69%    0.4209     1.74
Linear SVM              59.83%    0.5963     5.07
RBF SVM                 56.37%    0.3623     5.35
Decision Tree           54.23%    0.5411     0.20
Random Forest           58.26%    0.5814     0.94
AdaBoost                57.35%    0.4401     0.48
Multinomial NB          60.46%    0.5941     0.13
Best classifier by CV F1: Linear SVM (CV F1 = 0.5963)
Test Accuracy of best model: 61.51%
Test Macro-F1 of best model: 0.6127
